# Q5 — Logistic Regression on Iris Dataset
**Steps:**
1. Load Iris dataset
2. Filter only setosa and versicolor (binary classification)
3. Retrieve sepal_length and sepal_width as features
4. Normalize X
5. K-Fold Cross Validation (train-test split)
6. Train LogisticRegression, print w1, w2, b
7. Confusion matrix, Precision, Recall, F1
8. ROC Curve + AUC
9. Report Average Accuracy

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Import all required libraries
# ─────────────────────────────────────────────

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)
from mlxtend.plotting import plot_confusion_matrix

print('All libraries imported successfully!')

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Step 1: Load Iris Dataset
# ─────────────────────────────────────────────

# Load the iris dataset from seaborn
# It has 150 rows, 5 columns: sepal_length, sepal_width,
# petal_length, petal_width, species
iris = sns.load_dataset('iris')

print('Shape of full dataset:', iris.shape)
print('\nFirst 5 rows:')
iris.head()

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Step 2: Filter only setosa and versicolor
# The question says: Y Column should be numeric (0 or 1)
# setosa → 0, versicolor → 1
# ─────────────────────────────────────────────

# Keep only rows where species is setosa OR versicolor
iris_binary = iris[iris['species'].isin(['setosa', 'versicolor'])]

print('Shape after filtering:', iris_binary.shape)
print('\nClass distribution:')
print(iris_binary['species'].value_counts())

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Step 3 & 4: Retrieve features X and target Y
# Question says: use sepal_length and sepal_width as features
# ─────────────────────────────────────────────

# X = feature matrix (only 2 features as per question)
X = iris_binary[['sepal_length', 'sepal_width']].values

# y = target labels encoded as numbers (0 and 1)
# LabelEncoder converts: setosa → 0, versicolor → 1 (alphabetical order)
le = LabelEncoder()
y = le.fit_transform(iris_binary['species'])

print('X shape:', X.shape)   # should be (100, 2)
print('y shape:', y.shape)   # should be (100,)
print('\nClass mapping:', dict(zip(le.classes_, le.transform(le.classes_))))
print('\nFirst 5 X values:\n', X[:5])
print('\nFirst 5 y values:', y[:5])

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Step 5: Normalize the X part
# StandardScaler makes each feature have mean=0 and std=1
# This ensures no single feature dominates the model
# ─────────────────────────────────────────────

scaler = StandardScaler()

# fit_transform: learns mean & std FROM the data, then scales it
X = scaler.fit_transform(X)

print('After normalization:')
print('Mean of each feature (should be ~0):', X.mean(axis=0).round(4))
print('Std  of each feature (should be ~1):', X.std(axis=0).round(4))
print('\nFirst 5 normalized X values:\n', X[:5].round(4))

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Step 6: K-Fold Cross Validation Setup
# n_splits=5 means data is divided into 5 equal parts
# Each part takes a turn as test set (5 iterations total)
# shuffle=True: randomize order before splitting
# random_state=42: ensures same shuffle every run (reproducible)
# ─────────────────────────────────────────────

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store accuracy from each fold
train_accuracies = []
test_accuracies  = []

print('K-Fold setup done: 5 folds, shuffle=True, random_state=42')
print(f'Each fold: ~{100//5} samples for TEST, ~{100 - 100//5} for TRAIN')

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Main K-Fold Loop
# For each fold:
#   - Split into train/test
#   - Train LogisticRegression
#   - Print learned parameters w1, w2, b
#   - Compute and store train & test accuracy
#   - Plot confusion matrix
#   - Print classification report
#   - Plot ROC curve
# ─────────────────────────────────────────────

# We'll collect ROC data for all folds to plot together at end
all_fpr = []
all_tpr = []
all_auc = []

for fold_num, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    print('=' * 60)
    print(f'  FOLD {fold_num}')
    print('=' * 60)

    # ── Split data for this fold ──
    # train_idx: indices of rows used for training
    # test_idx : indices of rows used for testing
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    print(f'  Train size: {len(X_train)}, Test size: {len(X_test)}')

    # ── Train the Logistic Regression model ──
    # max_iter=1000 avoids convergence warnings on small data
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)   # learns from training data

    # ── Print learned parameters ──
    # model.coef_  : weights [w1, w2] for each feature
    # model.intercept_: bias term b
    # Decision boundary: w1*x1 + w2*x2 + b = 0
    w1, w2 = model.coef_[0]
    b       = model.intercept_[0]
    print(f'  Learned Parameters → w1: {w1:.4f}, w2: {w2:.4f}, b: {b:.4f}')

    # ── Predict on test and train sets ──
    y_test_pred  = model.predict(X_test)
    y_train_pred = model.predict(X_train)

    # ── Compute accuracies ──
    test_acc  = accuracy_score(y_test,  y_test_pred)
    train_acc = accuracy_score(y_train, y_train_pred)

    test_accuracies.append(test_acc)
    train_accuracies.append(train_acc)

    print(f'  TEST  accuracy: {test_acc:.4f}')
    print(f'  TRAIN accuracy: {train_acc:.4f}')

    # ── Confusion Matrix ──
    # confusion_matrix returns a 2x2 matrix:
    #   [[TN, FP],
    #    [FN, TP]]
    # TN = correctly predicted 0, TP = correctly predicted 1
    # FP = predicted 1 but actually 0, FN = predicted 0 but actually 1
    cm = confusion_matrix(y_test, y_test_pred)

    fig, ax = plot_confusion_matrix(
        conf_mat=cm,
        class_names=['setosa (0)', 'versicolor (1)'],
        show_absolute=True,
        show_normed=True,
        colorbar=True
    )
    plt.title(f'Confusion Matrix — Fold {fold_num}')
    plt.tight_layout()
    plt.show()

    # ── Classification Report ──
    # Shows Precision, Recall, F1-score for each class
    # Precision = TP / (TP + FP)  → of all predicted positives, how many were correct
    # Recall    = TP / (TP + FN)  → of all actual positives, how many did we catch
    # F1        = harmonic mean of precision and recall
    print(f'\n  Classification Report (Fold {fold_num}):')
    print(classification_report(
        y_test, y_test_pred,
        target_names=['setosa', 'versicolor']
    ))

    # ── ROC Curve data for this fold ──
    # predict_proba gives probability of class 0 and class 1
    # [:, 1] takes only the probability of being class 1 (versicolor)
    # roc_curve returns false positive rate, true positive rate at various thresholds
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score   = roc_auc_score(y_test, y_prob)

    all_fpr.append(fpr)
    all_tpr.append(tpr)
    all_auc.append(auc_score)

    print(f'  AUC Score: {auc_score:.4f}')
    print()

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — ROC Curve Plot (all 5 folds together)
# ROC curve plots:
#   X-axis: False Positive Rate (FPR) = FP / (FP + TN)
#   Y-axis: True  Positive Rate (TPR) = TP / (TP + FN)
# A perfect model goes to top-left corner → AUC = 1.0
# A random model is the diagonal line     → AUC = 0.5
# ─────────────────────────────────────────────

plt.figure(figsize=(8, 6))

colors = ['blue', 'green', 'red', 'orange', 'purple']

for i in range(5):
    plt.plot(
        all_fpr[i], all_tpr[i],
        color=colors[i],
        lw=2,
        label=f'Fold {i+1} (AUC = {all_auc[i]:.2f})'
    )

# Diagonal line = random classifier (baseline)
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve — Logistic Regression (All 5 Folds)')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Mean AUC across all folds: {np.mean(all_auc):.4f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Step 9: Report AVERAGE ACCURACY
# This is the final answer the question asks for
# Average across all 5 folds gives a reliable estimate
# of how well the model generalizes to unseen data
# ─────────────────────────────────────────────

avg_test_acc  = np.mean(test_accuracies)
avg_train_acc = np.mean(train_accuracies)

print('─' * 40)
print('  FINAL RESULTS')
print('─' * 40)
print()
print('  Per-fold TEST accuracies:')
for i, acc in enumerate(test_accuracies, 1):
    print(f'    Fold {i}: {acc:.4f}')

print()
print('  Per-fold TRAIN accuracies:')
for i, acc in enumerate(train_accuracies, 1):
    print(f'    Fold {i}: {acc:.4f}')

print()
print(f'  ★ AVERAGE TEST  ACCURACY: {avg_test_acc:.4f}  ({avg_test_acc*100:.2f}%)')
print(f'  ★ AVERAGE TRAIN ACCURACY: {avg_train_acc:.4f}  ({avg_train_acc*100:.2f}%)')
print()

# Check for overfitting:
# If train accuracy >> test accuracy, model is overfitting (memorizing training data)
# If both are similar, model generalizes well
diff = avg_train_acc - avg_test_acc
if diff < 0.05:
    print('  Model is NOT overfitting (train ≈ test accuracy) ✓')
else:
    print(f'  Warning: Train accuracy is {diff*100:.1f}% higher — possible overfitting')

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — [OPTIONAL] Decision Boundary Plot
# Visualizes what the model learned:
# The line where the model switches from predicting
# setosa (0) to predicting versicolor (1)
# We train on the full data just for visualization
# ─────────────────────────────────────────────

# Train a final model on all data for visualization
final_model = LogisticRegression(max_iter=1000)
final_model.fit(X, y)

# Create a mesh grid across the feature space
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

# Predict class for every point in the grid
Z = final_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))

# Colored regions show what the model predicts there
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')

# Scatter the actual data points
scatter = plt.scatter(
    X[:, 0], X[:, 1],
    c=y, cmap='coolwarm',
    edgecolors='black', linewidths=0.5, s=60
)

plt.xlabel('sepal_length (normalized)')
plt.ylabel('sepal_width (normalized)')
plt.title('Decision Boundary — Logistic Regression\nsetosa (blue) vs versicolor (red)')
plt.colorbar(scatter, label='Class (0=setosa, 1=versicolor)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('The boundary line is where: w1*x1 + w2*x2 + b = 0')
w1, w2 = final_model.coef_[0]
b = final_model.intercept_[0]
print(f'Equation: {w1:.4f}*sepal_length + {w2:.4f}*sepal_width + {b:.4f} = 0')